# 01 — Exploratory Data Analysis
### CodeAlpha Sentiment Analysis Project

This notebook explores the three source datasets (Amazon reviews, Twitter
Airline Sentiment, Financial PhraseBank) before any modeling: class
balance, text length, vocabulary, and domain-specific quirks. Findings
here directly informed the preprocessing and modeling decisions in
`docs/methodology.md`.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_all_domains
from src import config

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)


## Load and unify all three domains

In [ ]:
df = load_all_domains()
print(f"Total rows: {len(df):,}")
df.head(10)


## Class balance — overall and by domain

Sentiment analysis performance is strongly affected by class imbalance.
We expect financial text to skew neutral, and reviews to skew positive
(most people who bother leaving a review are happy customers).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

order = list(config.LABELS)
sns.countplot(data=df, x="label", order=order, hue="label", palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("Overall class distribution")

domain_counts = df.groupby(["domain", "label"]).size().unstack(fill_value=0).reindex(columns=order, fill_value=0)
domain_counts.plot(kind="bar", stacked=True, ax=axes[1], colormap="viridis")
axes[1].set_title("Class distribution by domain")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print(domain_counts)


## Text length distribution by domain

In [ ]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)

fig, ax = plt.subplots(figsize=(8, 4.5))
for domain in config.DOMAINS:
    subset = df[df["domain"] == domain]["word_count"]
    if len(subset):
        sns.kdeplot(subset, label=domain, fill=True, alpha=0.25, ax=ax)
ax.set_xlim(left=0)
ax.set_xlabel("Word count")
ax.set_title("Text length distribution by domain")
ax.legend()
plt.tight_layout()
plt.show()

df.groupby("domain")["word_count"].describe()


## Sample text per domain and label

Spot-checking raw examples is the fastest way to catch labeling issues
or domain quirks before they propagate into modeling.

In [ ]:
for domain in config.DOMAINS:
    print(f"\n{'='*70}\nDOMAIN: {domain}\n{'='*70}")
    for label in config.LABELS:
        sample = df[(df["domain"] == domain) & (df["label"] == label)]
        if len(sample):
            print(f"\n[{label.upper()}]")
            print(" -", sample.sample(1, random_state=1)["text"].values[0][:160])


## Most common words per domain (raw, pre-cleaning)

A quick unfiltered word frequency check — useful for spotting
domain-specific stopwords or artifacts (e.g. `flight`, `amazon`, `co`
from truncated URLs) that generic preprocessing might miss.

In [ ]:
from collections import Counter
import re

for domain in config.DOMAINS:
    text_blob = " ".join(df[df["domain"] == domain]["text"].astype(str).str.lower())
    words = re.findall(r"[a-z]{3,}", text_blob)
    top = Counter(words).most_common(15)
    print(f"\n{domain}: {top}")


## Key takeaways

- Record your actual findings here after running this notebook on the
  real downloaded datasets, e.g.:
  - Class balance: which domain is most/least balanced?
  - Text length: which domain has the shortest/longest texts, and does
    that suggest different preprocessing (e.g. tweets need less
    aggressive stopword removal since they're already short)?
  - Anything in the sample text that suggests a labeling quirk to watch
    for during modeling (e.g. sarcasm, mixed-sentiment reviews)?

These findings feed directly into the preprocessing and modeling choices
documented in `docs/methodology.md`.